In [1]:
from pathlib import Path
import yaml
import torch
from ultralytics import YOLO


## 1. Configuration

In [2]:
ROOT = Path(r'D:\PCB Defect Detection')
DATA_DIR = ROOT / 'Data' / 'processed' / 'none_640'
DATA_YAML = DATA_DIR / 'data.yaml'
RUNS_DIR = ROOT / 'runs'

PRETRAINED = ROOT / 'yolov8n.pt'
STAGE1_BEST = RUNS_DIR / 'yolov8n_640_frozen' / 'weights' / 'best.pt'
STAGE2_BEST = RUNS_DIR / 'yolov8n_640_finetune' / 'weights' / 'best.pt'

IMG_SIZE = 640
BATCH = 4
WORKERS = 0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42

assert DATA_DIR.exists(), DATA_DIR
assert DATA_YAML.exists(), DATA_YAML

with open(DATA_YAML, encoding='utf-8') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = str(DATA_DIR)
with open(DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_cfg, f, sort_keys=False, allow_unicode=True)

print(data_cfg['names'])
print('Device:', DEVICE)

{0: 'open', 1: 'short', 2: 'mousebite', 3: 'spur', 4: 'copper', 5: 'pin-hole'}
Device: cpu


## 2. Stage 1 — Frozen-backbone transfer learning

In [3]:
if not STAGE1_BEST.exists():
    model = YOLO(str(PRETRAINED))
    model.train(
        data=str(DATA_YAML), imgsz=IMG_SIZE, epochs=50, batch=BATCH,
        device=DEVICE, workers=WORKERS,
        optimizer='AdamW', lr0=1e-3, lrf=1e-2, weight_decay=5e-4,
        warmup_epochs=3, warmup_bias_lr=0.1,
        freeze=10, patience=10, seed=SEED,
        pretrained=True, amp=False,
        mosaic=0.0, close_mosaic=0,
        val=True, plots=True, save=True,
        project=str(RUNS_DIR), name='yolov8n_640_frozen', exist_ok=False
    )
else:
    print('Using existing:', STAGE1_BEST)

Using existing: D:\PCB Defect Detection\runs\yolov8n_640_frozen\weights\best.pt


## 3. Stage 2 — Full fine-tuning

In [4]:
assert STAGE1_BEST.exists(), STAGE1_BEST

model = YOLO(str(STAGE1_BEST))
model.train(
    data=str(DATA_YAML), imgsz=IMG_SIZE, epochs=30, batch=BATCH,
    device=DEVICE, workers=WORKERS,
    optimizer='AdamW', lr0=1e-4, lrf=1e-2, weight_decay=5e-4,
    warmup_epochs=2, warmup_bias_lr=0.0,
    freeze=0, patience=10, seed=SEED,
    pretrained=False, amp=False,
    mosaic=0.0, close_mosaic=0,
    val=True, plots=True, save=True,
    project=str(RUNS_DIR), name='yolov8n_640_finetune', exist_ok=False
)

Ultralytics 8.4.174  Python-3.13.5 torch-2.14.0+cpu CPU (12th Gen Intel Core i7-1260P)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=0, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=D:\PCB Defect Detection\Data\processed\none_640\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=D:\PCB Defect Detection\runs\yolov8n_640_frozen\weights\best.pt, momentum=0.937, mosaic=0.

2026/10/07 22:32:10 WARNING mlflow.utils.autologging_utils: MLflow spark autologging is known to be compatible with 3.1.2 <= pyspark <= 3.5.3, but the installed version is 3.5.4. If you encounter errors during autologging, try upgrading / downgrading pyspark to a compatible version, or try upgrading MLflow.
2026/10/07 22:32:10 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/10/07 22:32:10 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(ea66a66baccb455f9d73880199b0f377) to runs\mlflow
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri runs\mlflow'
MLflow: disable with 'yolo settings mlflow=False'
Using 1200 train, 150 val images for fraction=1.0 at imgsz=640
Using 0 dataloader workers
Logging results to D:\PCB Defect Detection\runs\yolov8n_640_finetune
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/30         0G      5.808      6.889       4.27         29        640: 100% ━━━━━━━━━━━━ 300/300 2.5s/it 12:251.9ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 1.9s/it 35.6s1.6ss
                   all        150       1005    0.00359    0.00265   0.000292   7.45e-05

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30         0G      4.681      5.605      3.933         29        640: 100% ━━━━━

KeyboardInterrupt: 

## 4. Validation

In [ ]:
assert STAGE2_BEST.exists(), STAGE2_BEST
best_model = YOLO(str(STAGE2_BEST))

val_metrics = best_model.val(
    data=str(DATA_YAML), split='val', imgsz=IMG_SIZE,
    batch=BATCH, device=DEVICE, workers=WORKERS, plots=True
)

p, r = val_metrics.box.mp, val_metrics.box.mr
f1 = 2*p*r/(p+r) if p+r else 0
print(f'Precision: {p:.4f}')
print(f'Recall:    {r:.4f}')
print(f'F1:        {f1:.4f}')
print(f'mAP@50:    {val_metrics.box.map50:.4f}')
print(f'mAP@50:95: {val_metrics.box.map:.4f}')

## 5. Final test evaluation

In [ ]:
test_metrics = best_model.val(
    data=str(DATA_YAML), split='test', imgsz=IMG_SIZE,
    batch=BATCH, device=DEVICE, workers=WORKERS, plots=True
)

p, r = test_metrics.box.mp, test_metrics.box.mr
f1 = 2*p*r/(p+r) if p+r else 0

print(f'Precision: {p:.4f}')
print(f'Recall:    {r:.4f}')
print(f'F1:        {f1:.4f}')
print(f'mAP@50:    {test_metrics.box.map50:.4f}')
print(f'mAP@50:95: {test_metrics.box.map:.4f}')

## 6. Per-class AP + model complexity

In [ ]:
for i, ap50 in enumerate(test_metrics.box.ap50):
    print(f'{best_model.names[i]:20s}: {ap50:.4f}')

params = sum(p.numel() for p in best_model.model.parameters())
inference_ms = test_metrics.speed.get('inference')
fps = 1000 / inference_ms if inference_ms else None

print(f'\nParameters: {params:,}')
if fps:
    print(f'Inference: {inference_ms:.2f} ms/image')
    print(f'FPS:       {fps:.2f}')

## 7. Test predictions

In [ ]:
TEST_IMAGES = DATA_DIR / 'test' / 'images'

best_model.predict(
    source=str(TEST_IMAGES), imgsz=IMG_SIZE, conf=0.25,
    device=DEVICE, save=True,
    project=str(RUNS_DIR), name='test_predictions', exist_ok=True
)

print(RUNS_DIR / 'test_predictions')